# 02 - Preprocessing

This notebook prepares the dataset for training.

It converts the raw file:

```text
../data/raw/cicids2017_original.csv
```

into the normalized file:

```text
../data/processed/soc_threat_hunting_input.csv
```

## What it does

- Cleans up column names.
- Replaces infinite values.
- Fills in missing (null) values.
- Creates derived SOC fields.
- Creates a binary `label`.
- Creates an `attack_type` column.
- Creates a `severity` column.
- Creates a `recommended_playbook` column.
- Creates basic MITRE ATT&CK mappings.
- Exports the processed dataset.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 1. Initial Setup

In [ ]:
# Libraries needed for data transformation.

import pandas as pd
import numpy as np
from pathlib import Path

# PATH_LOCAL_RAW = "../data/raw/"
# PATH_LOCAL_PROCESS = "../data/processed/"

RAW_PATH = Path("/content/drive/MyDrive/Proyecto final/threat-hunting-soc-ai/data/raw/cicids2017_original.csv")
OUTPUT_PATH = Path("/content/drive/MyDrive/Proyecto final/threat-hunting-soc-ai/data/processed/soc_threat_hunting_input.csv")

OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

print("Raw path:", RAW_PATH)
print("Raw exists:", RAW_PATH.exists())
print("Output path:", OUTPUT_PATH)

Raw path: /content/drive/MyDrive/Proyecto final/threat-hunting-soc-ai/data/raw/cicids2017_original.csv
Raw exists: True
Output path: /content/drive/MyDrive/Proyecto final/threat-hunting-soc-ai/data/processed/soc_threat_hunting_input.csv


## 2. Load Raw File

In [ ]:
# Load the raw dataset.
# Clean up spaces in column names to avoid access errors.

df = pd.read_csv(RAW_PATH)
df.columns = df.columns.str.strip()

df.head()

,Timestamp,Source IP,Destination IP,Source Port,Destination Port,Protocol,Flow Duration,Total Fwd Packets,Total Backward Packets,Total Length of Fwd Packets,...,Flow Bytes/s,Flow Packets/s,Packet Length Mean,Packet Length Std,FIN Flag Count,SYN Flag Count,RST Flag Count,PSH Flag Count,ACK Flag Count,Label
0,2026-06-20 08:14:20,192.168.2.108,81.188.20.104,7289,443,6,1009890,77,25,8472,...,10156.5517,101.0011,100.5588,35.8378,1,1,1,3,12,BENIGN
1,2026-06-20 13:48:59,192.168.1.23,98.48.218.60,19966,25,6,1443257,62,16,339,...,13550.6012,54.0444,250.7308,84.5561,1,2,1,2,4,BENIGN
2,2026-06-20 11:09:54,192.168.0.52,144.20.72.168,22801,248,6,11127,1,3,53,...,26422.2162,359.4859,73.5000,24.9005,1,6,4,0,3,PortScan
3,2026-06-20 15:32:12,192.168.3.176,44.205.80.165,2609,443,6,1783415,6,3,1171,...,7246.7709,5.0465,1436.0000,498.7867,1,1,1,1,13,BENIGN
4,2026-06-20 09:54:33,192.168.3.163,211.189.227.15,33630,22,6,585822,81,85,27851,...,52690.0663,283.3625,185.9458,69.0006,1,14,8,15,52,SSH-Patator


## 3. Basic Cleaning

In [ ]:
# Replace infinite values with NaN.
# Then fill NaN with 0 to prevent model errors.

df = df.replace([np.inf, -np.inf], np.nan)

print("Null values before fill:", df.isnull().sum().sum())

df = df.fillna(0)

print("Null values after fill:", df.isnull().sum().sum())

Null values before fill: 0
Null values after fill: 0


## 4. SOC Mapping Functions

In [ ]:
# These functions convert the original dataset label
# into useful fields for SOC operations and model training.

def map_label(label: str) -> str:
    # Binary classification for the first MVP.
    return "BENIGN" if str(label).upper() == "BENIGN" else "MALICIOUS"

def map_severity(attack_type: str) -> str:
    # Simple severity rule based on the attack type.
    # In a real SOC, this should consider asset criticality,
    # frequency, impact, and context.
    attack = str(attack_type).lower()

    if attack == "benign":
        return "Low"

    if "ddos" in attack or "dos" in attack or "bot" in attack:
        return "High"

    if "patator" in attack or "portscan" in attack:
        return "Medium"

    return "Medium"

def map_playbook(attack_type: str) -> str:
    # Maps the attack type to a recommended playbook.
    # This layer connects model detection with a SOC action.
    attack = str(attack_type).lower()

    if attack == "benign":
        return "no_action_required"

    if "ddos" in attack or "dos" in attack:
        return "dos_mitigation_playbook"

    if "bot" in attack:
        return "botnet_containment_playbook"

    if "patator" in attack:
        return "credential_attack_playbook"

    if "portscan" in attack:
        return "network_reconnaissance_playbook"

    return "generic_incident_response_playbook"

def map_mitre_tactic(attack_type: str) -> str:
    # Basic mapping to MITRE ATT&CK tactics.
    mapping = {
        "BENIGN": "",
        "PortScan": "Reconnaissance",
        "DDoS": "Impact",
        "DoS Hulk": "Impact",
        "Bot": "Command and Control",
        "FTP-Patator": "Credential Access",
        "SSH-Patator": "Credential Access",
    }

    return mapping.get(str(attack_type), "")

def map_mitre_technique(attack_type: str) -> str:
    # Basic mapping to MITRE ATT&CK techniques.
    mapping = {
        "BENIGN": "",
        "PortScan": "T1046",
        "DDoS": "T1498",
        "DoS Hulk": "T1499",
        "Bot": "T1071",
        "FTP-Patator": "T1110",
        "SSH-Patator": "T1110",
    }

    return mapping.get(str(attack_type), "")

## 5. Build Normalized Dataset

In [ ]:
# Create a new DataFrame with the standard project header.
# This header allows both the model and the SOC to work with a common format.

model_ready = pd.DataFrame()

model_ready["event_id"] = [f"EVT-{str(i + 1).zfill(6)}" for i in range(len(df))]
model_ready["timestamp"] = df.get("Timestamp", "")
model_ready["source_ip"] = df.get("Source IP", "")
model_ready["destination_ip"] = df.get("Destination IP", "")
model_ready["source_port"] = df.get("Source Port", 0)
model_ready["destination_port"] = df.get("Destination Port", 0)
model_ready["protocol"] = df.get("Protocol", 0)
model_ready["flow_duration"] = df.get("Flow Duration", 0)
model_ready["total_fwd_packets"] = df.get("Total Fwd Packets", 0)
model_ready["total_bwd_packets"] = df.get("Total Backward Packets", 0)
model_ready["total_length_fwd_packets"] = df.get("Total Length of Fwd Packets", 0)
model_ready["total_length_bwd_packets"] = df.get("Total Length of Bwd Packets", 0)
model_ready["flow_bytes_per_second"] = df.get("Flow Bytes/s", 0)
model_ready["flow_packets_per_second"] = df.get("Flow Packets/s", 0)
model_ready["packet_length_mean"] = df.get("Packet Length Mean", 0)
model_ready["packet_length_std"] = df.get("Packet Length Std", 0)

# Consolidate TCP flags into a single simple variable for the MVP.
flag_columns = [
    "FIN Flag Count",
    "SYN Flag Count",
    "RST Flag Count",
    "PSH Flag Count",
    "ACK Flag Count"
]

existing_flag_columns = [column for column in flag_columns if column in df.columns]

if existing_flag_columns:
    model_ready["tcp_flag_count"] = df[existing_flag_columns].sum(axis=1)
else:
    model_ready["tcp_flag_count"] = 0

# SOC context fields.
model_ready["event_source"] = "IDS"
model_ready["hostname"] = ""
model_ready["username"] = ""
model_ready["process_name"] = ""
model_ready["alert_signature"] = df.get("Label", "")

# MITRE fields.
model_ready["mitre_tactic"] = df["Label"].apply(map_mitre_tactic)
model_ready["mitre_technique"] = df["Label"].apply(map_mitre_technique)

# Target and response fields.
model_ready["label"] = df["Label"].apply(map_label)
model_ready["attack_type"] = df["Label"]
model_ready["severity"] = df["Label"].apply(map_severity)
model_ready["recommended_playbook"] = df["Label"].apply(map_playbook)

model_ready.head()

,event_id,timestamp,source_ip,destination_ip,source_port,destination_port,protocol,flow_duration,total_fwd_packets,total_bwd_packets,...,hostname,username,process_name,alert_signature,mitre_tactic,mitre_technique,label,attack_type,severity,recommended_playbook
0,EVT-000001,2026-06-20 08:14:20,192.168.2.108,81.188.20.104,7289,443,6,1009890,77,25,...,,,,BENIGN,,,BENIGN,BENIGN,Low,no_action_required
1,EVT-000002,2026-06-20 13:48:59,192.168.1.23,98.48.218.60,19966,25,6,1443257,62,16,...,,,,BENIGN,,,BENIGN,BENIGN,Low,no_action_required
2,EVT-000003,2026-06-20 11:09:54,192.168.0.52,144.20.72.168,22801,248,6,11127,1,3,...,,,,PortScan,Reconnaissance,T1046,MALICIOUS,PortScan,Medium,network_reconnaissance_playbook
3,EVT-000004,2026-06-20 15:32:12,192.168.3.176,44.205.80.165,2609,443,6,1783415,6,3,...,,,,BENIGN,,,BENIGN,BENIGN,Low,no_action_required
4,EVT-000005,2026-06-20 09:54:33,192.168.3.163,211.189.227.15,33630,22,6,585822,81,85,...,,,,SSH-Patator,Credential Access,T1110,MALICIOUS,SSH-Patator,Medium,credential_attack_playbook


## 6. Validation of the Processed Dataset

In [ ]:
# Validate size, label distribution, attack types, and playbooks.

print("Rows:", model_ready.shape[0])
print("Columns:", model_ready.shape[1])

print("\nLabels:")
print(model_ready["label"].value_counts())

print("\nAttack types:")
print(model_ready["attack_type"].value_counts())

print("\nSeverity:")
print(model_ready["severity"].value_counts())

print("\nRecommended playbooks:")
print(model_ready["recommended_playbook"].value_counts())

Rows: 1000
Columns: 28

Labels:
label
BENIGN       543
MALICIOUS    457
Name: count, dtype: int64

Attack types:
attack_type
BENIGN         543
PortScan       123
DDoS            92
DoS Hulk        90
Bot             71
SSH-Patator     47
FTP-Patator     34
Name: count, dtype: int64

Severity:
severity
Low       543
High      253
Medium    204
Name: count, dtype: int64

Recommended playbooks:
recommended_playbook
no_action_required                 543
dos_mitigation_playbook            182
network_reconnaissance_playbook    123
credential_attack_playbook          81
botnet_containment_playbook         71
Name: count, dtype: int64


## 7. Validation of Final Columns

In [ ]:
# Confirm that the file contains all expected columns.

expected_columns = [
    "event_id",
    "timestamp",
    "source_ip",
    "destination_ip",
    "source_port",
    "destination_port",
    "protocol",
    "flow_duration",
    "total_fwd_packets",
    "total_bwd_packets",
    "total_length_fwd_packets",
    "total_length_bwd_packets",
    "flow_bytes_per_second",
    "flow_packets_per_second",
    "packet_length_mean",
    "packet_length_std",
    "tcp_flag_count",
    "event_source",
    "hostname",
    "username",
    "process_name",
    "alert_signature",
    "mitre_tactic",
    "mitre_technique",
    "label",
    "attack_type",
    "severity",
    "recommended_playbook"
]

missing_columns = [column for column in expected_columns if column not in model_ready.columns]

if missing_columns:
    print("Missing columns:", missing_columns)
else:
    print("All expected columns are present.")

Todas las columnas esperadas están presentes.


## 8. Export Processed Dataset

In [ ]:
# Save the processed file.
# This file will be the input for the training notebooks.

model_ready.to_csv(OUTPUT_PATH, index=False)

print("Processed dataset saved at:", OUTPUT_PATH)

Processed dataset saved at: /content/drive/MyDrive/Proyecto final/threat-hunting-soc-ai/data/processed/soc_threat_hunting_input.csv


## 9. Preprocessing Conclusions

The dataset was normalized to the standard project layout.  
It now includes technical network variables and derived SOC fields.

The generated file will be used by:

```text
03_baseline_model.ipynb
04_deep_learning_model.ipynb
```
